In [ ]:
from google.colab import drive
import sys
import os

#ドライブをマウント
drive.mount('/content/drive')
# My Driveのパスをシステムパスに追加します
my_drive_path = '/content/drive/MyDrive'
if my_drive_path not in sys.path:
    sys.path.append(my_drive_path)

print("sys.path に My Drive を追加しました。")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
sys.path に My Drive を追加しました。


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets, transforms
from datasets import load_dataset
import random
from transformers import CLIPTokenizer, CLIPConfig
from CLIP.model import CLIP, CLIPClassifier
from CLIP.engine import clip_training

In [ ]:
# 1. アノテーション（テキスト）と画像（Train 2017の一部）を落とす
!wget -q http://images.cocodataset.org/annotations/annotations_trainval2017.zip
!unzip -q annotations_trainval2017.zip

# 2. 画像は11万枚落とすと時間がかかるので、検証用に validation (5,000枚) を使う
# CLIPのスクラッチ実装の検証（Loss減少の確認）には5,000枚を使う
!wget -q http://images.cocodataset.org/zips/val2017.zip
!unzip -q val2017.zip

In [ ]:
#自作CLIP用に224×224にリサイズ
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    # CLIPの事前学習済み重み（ResNet/ViT）の多くがImageNetで学習されているため、
    # その統計値に合わせてNormalizeを行う。
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

#自作CLIPの動作確認用
coco_check = datasets.CocoCaptions(
    root = 'val2017',
    annFile = 'annotations/captions_val2017.json',
    transform = transform
)

loading annotations into memory...
Done (t=0.04s)
creating index...
index created!


In [ ]:
img, caption = coco_check[0]
print(f"画像サイズ: {img.shape}")
print(f"キャプション: {caption}")

画像サイズ: torch.Size([3, 224, 224])
キャプション: ['A woman stands in the dining area at the table.', 'A room with chairs, a table, and a woman in it.', 'A woman standing in a kitchen by a window', 'A person standing at a table in a room.', 'A living area with a television and a table']


In [ ]:
#キャプションを5つの中から1つ選び、トークン化するDataLoaderの作成
class CocoDataset(Dataset):
    def __init__(self, coco_dataset, tokenizer=None):
        self.coco = coco_dataset
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.coco)

    def __getitem__(self, n):
        img, captions = self.coco[n]

        #キャプションをランダムに選ぶ
        caption = random.choice(captions)

        #トークナイザーがある場合はトークン化
        if self.tokenizer:
            caption = "picture of" + caption
            caption = self.tokenizer(caption, padding="max_length", max_length=77, truncation=True, return_tensors='pt')
            return img, caption.input_ids.squeeze(0)
        return img, caption

In [ ]:
#OpenAIのCLIPトークナイザー、CLIPConfigを使用
tokenizer = CLIPTokenizer.from_pretrained("openai/clip-vit-base-patch32")
config = CLIPConfig.from_pretrained("openai/clip-vit-base-patch32")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

In [ ]:
coco_check_dataset = CocoDataset(coco_check, tokenizer=tokenizer)
train_loader = DataLoader(coco_check_dataset, batch_size=128, shuffle=True, num_workers=2, pin_memory=True)

In [ ]:
model = CLIP(3, config)

In [ ]:
#GPUに設定
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model.to(device=device)
loss_fn = nn.CrossEntropyLoss()

In [ ]:
optimizer = optim.AdamW(
    model.parameters(),
    #今回はバッチサイズが小さいため、学習率を小さく設定する
    lr=1e-4,
    weight_decay=0.1,
    betas=(0.9, 0.98),
    eps=1e-6
)

In [ ]:
clip_training(100, optimizer, loss_fn, train_loader, model, device)

Epoch: 1, Loss: 4.890561401844025
Epoch: 2, Loss: 4.783298581838608
Epoch: 3, Loss: 4.783004021644592
Epoch: 4, Loss: 4.782793271541595
Epoch: 5, Loss: 4.782918417453766
Epoch: 6, Loss: 4.782820379734039
Epoch: 7, Loss: 4.783060783147812
Epoch: 8, Loss: 4.78293662071228
Epoch: 9, Loss: 4.783118611574173
Epoch: 10, Loss: 4.774392712116241
Epoch: 11, Loss: 4.73795782327652
Epoch: 12, Loss: 4.7040659308433534
Epoch: 13, Loss: 4.686181348562241
Epoch: 14, Loss: 4.675025480985641
Epoch: 15, Loss: 4.626086232066155
Epoch: 16, Loss: 4.609638023376465
Epoch: 17, Loss: 4.58611244559288
Epoch: 18, Loss: 4.592645627260208
Epoch: 19, Loss: 4.554648584127426
Epoch: 20, Loss: 4.533100593090057
Epoch: 21, Loss: 4.540729427337647
Epoch: 22, Loss: 4.467138105630875
Epoch: 23, Loss: 4.4797833383083345
Epoch: 24, Loss: 4.447931632399559
Epoch: 25, Loss: 4.41915545463562
Epoch: 26, Loss: 4.413081759214402
Epoch: 27, Loss: 4.37133549451828
Epoch: 28, Loss: 4.348676073551178
Epoch: 29, Loss: 4.3242853403091